In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *
from imageutils import *
from preprocess import *
from scipy.ndimage import gaussian_filter
from fittingutils import *

In [26]:
folder = 'temp1/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp1/solo_L1_phi-fdt-ilam_20250310T080009_V202503131733C_0563100100.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T080608_V202503131733C_0563100125.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T081208_V202503131835C_0563100150.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T081808_V202503131935C_0563100175.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T082408_V202503131935C_0563100200.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T083008_V202503132033C_0563100225.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T083608_V202503141634C_0563100250.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T084208_V202503141734C_0563100275.fits',
 'temp1/solo_L1_phi-fdt-ilam_20250310T084808_V202503141833C_0563100300.fits']

In [27]:
flatfield = np.load('transmittance.npz')['flatfield']
flatfield = np.nan_to_num(flatfield, nan=1)

In [28]:
images = []

for file in files[:]:
    with fits.open(file) as hdul:
        data = hdul[0].data
        header = hdul[0].header
        data = data / flatfield
        data = realign(data, header=header)
        stokes = demodulate(data)

    images.append(stokes)

images = np.array(images)

In [5]:
def calc_reflection_center(I, Q):
    from scipy.ndimage import binary_dilation
    from skimage.feature import canny

    a = np.percentile(I[0], 0.1)
    b = np.percentile(I[0], 99.9)
    threshold = a + (b - a) * 0.1

    X, Y = [], []

    for i in range(len(images)):
        mask = binary_dilation(I[i] > threshold, iterations=5)
        edges = canny(Q[i], sigma=8, low_threshold=0.9, high_threshold=0.9, use_quantiles=True)
        edges *= ~mask

        xe, ye = np.where(edges)
        xe, ye = filter_outliers(xe, ye)

        xg, yg, rg = fitnp(xe, ye)
        xc, yc, rs = find_center(I[i])

        X.append((xc + xg) / 2)
        Y.append((yc + yg) / 2)

    return np.median(X), np.median(Y)

In [6]:
def remove_airy(data, size=255, niter=5):
    from astropy.modeling.functional_models import AiryDisk2D
    from scipy.signal import fftconvolve

    if len(data.shape) == 2:
        xi, yi = np.mgrid[-size:size + 1,-size:size + 1]
        airy = AiryDisk2D(radius=2.5)
        q = airy(xi, yi)
        q /= np.sum(q)

        result = data.copy()
        for _ in range(niter):
            temp = fftconvolve(result, q, mode='same')
            result = data - 1 * (temp - data)
        return result
    else:
        out = []
        for i in range(len(data)):
            out.append(remove_airy(data[i], size=size, niter=niter))
        return np.array(out)

In [29]:
I = images[:,0]
Q = images[:,1]
xr, yr = calc_reflection_center(I, Q)

In [32]:
I = images[:,0].copy()
I = remove_airy(I)
I_ = reflect(I, xr, yr)
#I_ = gaussian_filter(I_, 8, axes=(-2,-1))

mask = np.all(I_ < 1000, axis=0)

I_[I > 1000] = np.nan
I[I > 1000] = np.nan

In [34]:
xr, yr

(np.float64(979.9573416476331), np.float64(978.915311133283))

In [35]:
q = 0

I0 = I[2].copy()
I0[I_[2] > 1000] = np.nan
q_ = np.nanmean((I - I0) * I_, axis=0) / np.nanmean((I_ - I0 * 0) ** 2, axis=0)
q_[mask] = np.nan
q = polyfit2d(q_, degree=1)

plt.figure(figsize=(10,10))
plt.imshow(q_, vmin=0, vmax=15e-3)
plt.tight_layout()

/tmp/ipykernel_171816/423269062.py:5: RuntimeWarning: Mean of empty slice
  q_ = np.nanmean((I - I0) * I_, axis=0) / np.nanmean((I_ - I0 * 0) ** 2, axis=0)
/tmp/ipykernel_171816/423269062.py:5: RuntimeWarning: invalid value encountered in divide
  q_ = np.nanmean((I - I0) * I_, axis=0) / np.nanmean((I_ - I0 * 0) ** 2, axis=0)


In [36]:
plt.figure(figsize=(10,10))
plt.imshow(q_ - q, vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [11]:
q = 0

for _ in range(5):
    I0 = I[2] - q * I_[2]
    q_ = np.nanmean((I - I0) * I_, axis=0) / np.nanmean((I_ - I0 * 0) ** 2, axis=0)
    q_[mask] = np.nan
    q = polyfit2d(q_, degree=1)

plt.figure(figsize=(10,10))
plt.imshow(q, vmin=0, vmax=15e-3)
plt.tight_layout()

/tmp/ipykernel_171816/2691052332.py:5: RuntimeWarning: Mean of empty slice
  q_ = np.nanmean((I - I0) * I_, axis=0) / np.nanmean((I_ - I0 * 0) ** 2, axis=0)
/tmp/ipykernel_171816/2691052332.py:5: RuntimeWarning: invalid value encountered in divide
  q_ = np.nanmean((I - I0) * I_, axis=0) / np.nanmean((I_ - I0 * 0) ** 2, axis=0)


In [348]:
i = 4

plt.figure(figsize=(10,10))
plt.imshow(I[i] - q * I_[i], vmin=-100, vmax=200)
plt.tight_layout()

In [339]:
plt.figure(figsize=(10,10))
plt.imshow(I[i], vmin=-100, vmax=200)
plt.tight_layout()